## RAG with Azure Cosmos DB

![cosmosdb-rag](./Images/cosmosdb-rag.png)

### Installing Libraries and Utilities

In [ ]:
%pip install azure-cosmos==4.16.0 azure-identity python-dotenv openai==2.38.0

### Setting up the Environment

In [ ]:
import os 
from dotenv import load_dotenv

load_dotenv()

# fetching the cosmosdb configuration from environment variables
cosmosdb_endpoint = os.getenv("COSMOSDB_ENDPOINT")
cosmosdb_key = os.getenv("COSMOSDB_KEY")
database_name = os.getenv("DATABASE_NAME")
container_name = os.getenv("CONTAINER_NAME") + "Vector"

# fetching the azure openai configuration from environment variables
azure_openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_openai_key = os.getenv("AZURE_OPENAI_KEY")
embedding_model_name = os.getenv("EMBEDDING_MODEL_NAME")
chat_completions_model_name = os.getenv("CHAT_COMPLETIONS_MODEL_NAME")

### Create the Cosmos DB Client

In [ ]:
from azure.cosmos import CosmosClient
from azure.cosmos import PartitionKey

client = CosmosClient(cosmosdb_endpoint, cosmosdb_key)

### Navigate the Resource Hierarchy

In [ ]:
database = client.get_database_client(database_name)
container = database.get_container_client(container_name)

### Create the Azure OpenAI Client

In [ ]:
from openai import AzureOpenAI

azure_openai_client = AzureOpenAI(
    api_key=azure_openai_key,
    api_version="2024-02-15-preview",
    azure_endpoint=azure_openai_endpoint
)

### Create the Embedding Generator Helper Function

In [ ]:
def generate_embeddings(client, text):
    
    response = client.embeddings.create(
        input=text,
        model = embedding_model_name
    )
    
    embeddings=response.model_dump()
    return embeddings['data'][0]['embedding']
    

### Setting the User Query

Some user queries to try:
1) Recommend vegetarian dishes with high protein.
2) What low-carb foods would you recommend?
3) Suggest me something with mangoes and bananas in it.

In [ ]:
user_query = "Suggest me something with mangoes and bananas in it."

### Retrieve Context with Hybrid Search

In [ ]:
import json

search_query = """ SELECT TOP 5 VALUE {
    "id": c.id,
    "name": c.name,
    "content": c.content,

    "metadata": {
        "category": c.category,
        "description": c.description,
        "price": c.price,
        "restaurantId": c.restaurantId,
        "rating": c.rating,
        "reviewCount": c.reviewCount,
        "priceValue": c.priceValue,
        "currency": c.currency,
        "available": c.available,
        "calories": c.calories,
        "dietaryTags": c.dietaryTags
    }
}

FROM c

ORDER BY RANK RRF(
    VectorDistance(c.vector, @queryVector),
    FullTextScore(c.content, @queryText)
)
"""

query_vector = generate_embeddings(azure_openai_client, user_query)
parameters = [
    {"name": "@queryVector", "value": query_vector},
    {"name": "@queryText", "value": user_query}
]

context = {
    "retrieved_documents": list(
        container.query_items(
            query=search_query,
            parameters=parameters,
            enable_cross_partition_query=True
        )
    )
}

context_json = json.dumps(context, indent=2)

print(context_json)


### Setting the System Prompt for our RAG Agent

In [ ]:
system_prompt = """
You are FoodGPT, an intelligent restaurant recommendation assistant.

Your role is to answer user questions using ONLY the restaurant and menu information provided in the retrieved context.

Instructions:
- Carefully review the retrieved context before answering.
- Use only the information present in the retrieved context.
- Do not invent menu items, ingredients, prices, ratings, calories, dietary tags, or categories.
- If multiple menu items are relevant, explain why they match the user's request.
- When recommending items, prioritize the most relevant and highly rated options from the retrieved context.
- Mention useful details such as category, rating, price, calories, and dietary tags whenever available.
- If the retrieved context does not contain sufficient information to answer the question, clearly state that the information is unavailable.
- Never claim to know information that is not present in the retrieved context.
- Keep responses concise, helpful, and conversational.
- Format recommendations using bullet points when appropriate.

"""

### Make an API Call to the LLM

In [ ]:
user_message = f""" the user query is: {user_query}
the context is : {context_json}"""

chat_completions_response = azure_openai_client.chat.completions.create(
    model = chat_completions_model_name,
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message}
    ],
    temperature=0.7
)

print(chat_completions_response.choices[0].message.content)